In [1]:
pip install azure-ai-projects azure-core azure-storage-blob

   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 21.8 MB/s  0:00:00

   ---------------------------------------- 0/9 [PyJWT]
   ---- ----------------------------------- 1/9 [isodate]
   -------- ------------------------------- 2/9 [cryptography]
   -------- ------------------------------- 2/9 [cryptography]
   -------- ------------------------------- 2/9 [cryptography]
   -------- ------------------------------- 2/9 [cryptography]
   -------- ------------------------------- 2/9 [cryptography]
   -------- ------------------------------- 2/9 [cryptography]
   ------------- -------------------------- 3/9 [azure-core]
   ------------- -------------------------- 3/9 [azure-core]
   ------------- -------------------------- 3/9 [azure-core]
   ------------- -------------------------- 3/9 [azure-core]
   ------------- -------------------------- 3/9 [azure-core]
   ------------- -------------------------- 3/9 [azure-core

In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.2 MB 7.8 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 6.8 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.29.0
    Uninstalling openai-2.29.0:
      Successfully uninstalled openai-2.29.0
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
from openai import OpenAI
import os


client = OpenAI(
    api_key="",  # Picks up the secret key from Colab environment
    base_url="https://smartde.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [6]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": f"RESUME SOURCE:\n{resume_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": ["Project Management", "PR", "Leadership"]\n}', type='output_text', logprobs=[])]


In [7]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Name": "Richard Sanchez",
  "Education": "Master of Business Management (2029-2030)",
  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",
  "Top 3 Skills": [
    "Project Management",
    "PR",
    "Leadership"
  ]
}


In [ ]:
from azure.storage.blob import BlobServiceClient

# ✅ Correct connection string
AZURE_STORAGE_CONNECTION_STRING = ""

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "resumecontainer"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [10]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [11]:
blob_client = blob_service_client.get_blob_client(
    container="resumecontainer",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F6688CDB274"',
 'last_modified': datetime.datetime(2026, 4, 21, 5, 26, 26, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xdb\xe3g\xe1$0\xd2G\xd7k\xff\x97\xbd\xe3\xdb`'),
 'client_request_id': 'a490d969-3d42-11f1-af74-8d8e26b10c60',
 'request_id': '4b8ff440-601e-0034-2c4f-d1203c000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 5, 26, 25, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [12]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260421_105622.txt
